# Audio-to-Frame Mapping

Connects audio segments to visual frames for a chosen debate.

At 1 fps, frame N covers the time window **[N-1, N) seconds**, so:
- `frame_start = floor(time_stamp) + 1`
- `frame_end   = ceil(time_stamp + duration)` (last frame with any overlap)

**Change `DEBATE` below** to run on any debate video.

In [1]:
DEBATE        = 'Martins_vs_Gouveia_Melo_November_23'
FEATURES_DIR  = '../Project_Features'

In [2]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

## 1 — Load audio data and assign speaker labels

Speaker labels require the cross-debate pipeline: all 28 debates are loaded,
k=3 clusters are built per debate, global per-candidate centroids are derived,
then clusters are matched to candidate names. Condensed from `audio_candidate_analysis.ipynb`.

In [3]:
audio_files = sorted([f for f in os.listdir(FEATURES_DIR) if f.endswith('_audio.pkl')])

dfs = []
for f in audio_files:
    df = pd.read_pickle(os.path.join(FEATURES_DIR, f))
    df['video'] = f.replace('_audio.pkl', '')
    dfs.append(df)

data_audio = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(data_audio)} segments across {data_audio["video"].nunique()} videos')

Loaded 2656 segments across 28 videos


In [4]:
def parse_embedding(s):
    if isinstance(s, np.ndarray):
        return s
    return np.fromstring(s.strip('[]'), sep=' ')

data_audio['speak_embeddings'] = data_audio['speak_embeddings'].apply(parse_embedding)

data_parties       = pd.read_pickle(os.path.join(FEATURES_DIR, 'candidate_party.pkl'))
candidate_to_party = dict(zip(data_parties['Candidate'], data_parties['Party']))

CANDIDATES = [
    'Cotrim_Figueiredo', 'Filipe', 'Gouveia_Melo',
    'Marques_Mendes', 'Martins', 'Pinto', 'Seguro', 'Ventura'
]
debate_videos = sorted([v for v in data_audio['video'].unique() if 'vs' in v])

def extract_candidates(video_name):
    for c in CANDIDATES:
        if video_name.startswith(c):
            rest = video_name[len(c) + len('_vs_'):]
            for c2 in CANDIDATES:
                if rest.startswith(c2):
                    return c, c2
    return None, None

In [5]:
debate_centroids = {}
cluster_col      = np.full(len(data_audio), -1, dtype=int)

for video_name in debate_videos:
    mask = data_audio['video'] == video_name
    idx  = data_audio.index[mask]
    E    = np.stack(data_audio.loc[idx, 'speak_embeddings'].values)
    km   = KMeans(n_clusters=3, random_state=42, n_init=10)
    lbls = km.fit_predict(E)
    cluster_col[idx]              = lbls
    debate_centroids[video_name]  = km.cluster_centers_

data_audio['cluster_k3'] = cluster_col
print(f'k=3 clustering done for {len(debate_centroids)} debates')

k=3 clustering done for 28 debates


In [6]:
def candidate_centroid_per_debate(candidate, debate_centroids):
    their_debates = [v for v in debate_centroids if candidate in v]
    if len(their_debates) < 2:
        return {}
    best = {}
    for video in their_debates:
        others = [v for v in their_debates if v != video]
        cents  = debate_centroids[video]
        scores = [
            sum(cdist(cents[i].reshape(1,-1), debate_centroids[ov], metric='cosine')[0].min()
                for ov in others)
            for i in range(3)
        ]
        best[video] = cents[int(np.argmin(scores))]
    return best

global_centroids = {}
for cand in CANDIDATES:
    result = candidate_centroid_per_debate(cand, debate_centroids)
    if result:
        global_centroids[cand] = np.mean(np.stack(list(result.values())), axis=0)

print(f'Global centroids built for {len(global_centroids)} candidates')

Global centroids built for 8 candidates


In [7]:
speaker_col = np.full(len(data_audio), 'unknown', dtype=object)

for video_name in debate_videos:
    cA, cB = extract_candidates(video_name)
    if cA is None:
        continue
    mask  = data_audio['video'] == video_name
    idx   = data_audio.index[mask]
    cents = debate_centroids[video_name]

    dist_A = cdist(cents, global_centroids[cA].reshape(1,-1), metric='cosine').flatten()
    dist_B = cdist(cents, global_centroids[cB].reshape(1,-1), metric='cosine').flatten()

    cluster_map = {}
    available   = {0, 1, 2}
    best_A = int(np.argmin([dist_A[i] if i in available else np.inf for i in range(3)]))
    cluster_map[best_A] = cA
    available.remove(best_A)
    best_B = int(np.argmin([dist_B[i] if i in available else np.inf for i in range(3)]))
    cluster_map[best_B] = cB
    available.remove(best_B)
    cluster_map[available.pop()] = 'host'

    labels = data_audio.loc[idx, 'cluster_k3'].values
    speaker_col[idx] = [cluster_map[l] for l in labels]

data_audio['speaker'] = speaker_col
data_audio['party']   = data_audio['speaker'].map(candidate_to_party).fillna('host')

print('Speaker assignment done.')
print(data_audio[data_audio['video'] == DEBATE]['speaker'].value_counts().to_string())

Speaker assignment done.
speaker
Gouveia_Melo    57
Martins         26
host            16


## 2 — Load visual data for the chosen debate

In [8]:
visual_file = os.path.join(FEATURES_DIR, f'{DEBATE}_visual.pkl')
data_visual = pd.read_pickle(visual_file)

data_visual['frame_number'] = (
    data_visual['Frame']
    .str.extract(r'frame_(\d+)\.jpg')[0]
    .astype(int)
)
data_visual = data_visual.sort_values('frame_number').reset_index(drop=True)

def count_faces(face_list):
    if isinstance(face_list, list):
        return len(face_list)
    return 0

data_visual['face_count'] = data_visual['Fer'].apply(count_faces)

total_frames = len(data_visual)
print(f'Debate  : {DEBATE}')
print(f'Frames  : {total_frames}  (range {data_visual["frame_number"].min()}–{data_visual["frame_number"].max()})')
print(f'Face count distribution:')
print(data_visual['face_count'].value_counts().sort_index().to_string())

Debate  : Martins_vs_Gouveia_Melo_November_23
Frames  : 2048  (range 1–2048)
Face count distribution:
face_count
0      1
1    989
2    901
3    156
4      1


## 3 — Build audio-to-frame mapping

In [9]:
debate_audio = (
    data_audio[data_audio['video'] == DEBATE]
    .copy()
    .sort_values('time stamp')
    .reset_index(drop=True)
)

debate_audio['frame_start'] = (debate_audio['time stamp'].apply(math.floor) + 1).astype(int)
debate_audio['frame_end']   = (debate_audio['time stamp'] + debate_audio['duration']).apply(math.ceil).astype(int)
debate_audio['n_frames']    = debate_audio['frame_end'] - debate_audio['frame_start'] + 1
debate_audio['frames']      = debate_audio.apply(
    lambda r: list(range(r['frame_start'], r['frame_end'] + 1)), axis=1
)

print(f'Mapping built for {len(debate_audio)} audio segments.')

Mapping built for 99 audio segments.


## 4 — Summary statistics

In [10]:
all_audio_frames   = set(f for frames in debate_audio['frames'] for f in frames)
existing_frames    = set(data_visual['frame_number'].unique())
covered_frames     = all_audio_frames & existing_frames
not_covered_frames = existing_frames - all_audio_frames

speaker_stats = (
    debate_audio
    .groupby(['speaker', 'party'])
    .agg(
        segments         = ('time stamp', 'count'),
        total_duration_s = ('duration', 'sum'),
        total_frames     = ('n_frames', 'sum')
    )
    .reset_index()
    .sort_values('segments', ascending=False)
)

print('=' * 58)
print(f'  Debate : {DEBATE}')
print('=' * 58)
print(f'  Audio segments            : {len(debate_audio)}')
print(f'  Total frames (visual)     : {total_frames}')
print(f'  Frames covered by audio   : {len(covered_frames)}  ({len(covered_frames)/total_frames*100:.1f}%)')
print(f'  Frames NOT in any segment : {len(not_covered_frames)}  ({len(not_covered_frames)/total_frames*100:.1f}%)')
print()
print('Per-speaker breakdown:')
print(speaker_stats.to_string(index=False))

  Debate : Martins_vs_Gouveia_Melo_November_23
  Audio segments            : 99
  Total frames (visual)     : 2048
  Frames covered by audio   : 1795  (87.6%)
  Frames NOT in any segment : 253  (12.4%)

Per-speaker breakdown:
     speaker       party  segments  total_duration_s  total_frames
Gouveia_Melo Independent        57           647.696           705
     Martins          BE        26           882.731           912
        host        host        16           210.449           225


## 5 — Interleaved timeline: audio segments + gaps

Gap rows (yellow) show the frames between two consecutive audio segments,
how many frames they span, and the duration of the silence in seconds.

In [11]:
rows = []
for i in range(len(debate_audio)):
    seg = debate_audio.iloc[i]

    # ── gap before this segment ────────────────────────────────────────────
    if i > 0:
        prev      = debate_audio.iloc[i - 1]
        g_f_start = int(prev["frame_end"]) + 1
        g_f_end   = int(seg["frame_start"]) - 1
        if g_f_start <= g_f_end:
            gap_s = seg["time stamp"] - (prev["time stamp"] + prev["duration"])
            rows.append({
                "type"       : "gap",
                "segment"    : "",
                "t_start_s"  : "",
                "duration_s" : f"{gap_s:.2f}",
                "speaker"    : "(silence)",
                "party"      : "",
                "frame_start": g_f_start,
                "frame_end"  : g_f_end,
                "n_frames"   : g_f_end - g_f_start + 1,
            })

    # ── audio segment ─────────────────────────────────────────────────────
    rows.append({
        "type"       : "audio",
        "segment"    : i,
        "t_start_s"  : f"{seg['time stamp']:.3f}",
        "duration_s" : f"{seg['duration']:.3f}",
        "speaker"    : seg["speaker"],
        "party"      : seg["party"],
        "frame_start": int(seg["frame_start"]),
        "frame_end"  : int(seg["frame_end"]),
        "n_frames"   : int(seg["n_frames"]),
    })

timeline_df = pd.DataFrame(rows).reset_index(drop=True)
display_df  = timeline_df.drop(columns="type")

# _style_row gets a row from display_df (no type col); look it up in timeline_df by index
def _style_row(row):
    if timeline_df.loc[row.name, "type"] == "gap":
        return ["background-color: #fff8dc; color: #7a6000"] * len(row)
    return [""] * len(row)

(
    display_df
    .style
    .apply(_style_row, axis=1)
    .set_caption(f"Timeline — {DEBATE}  |  audio segments (white) + silence gaps (yellow)")
)

,segment,t_start_s,duration_s,speaker,party,frame_start,frame_end,n_frames
0,0,0.031,23.726,host,host,1,24,24
1,,,1.32,(silence),,25,25,1
2,1,25.073,16.048,Gouveia_Melo,Independent,26,42,17
3,2,42.455,9.551,Gouveia_Melo,Independent,43,53,11
4,3,53.187,6.902,Gouveia_Melo,Independent,54,61,8
5,4,60.764,18.411,Gouveia_Melo,Independent,61,80,20
6,5,79.647,16.132,Gouveia_Melo,Independent,80,96,17
7,6,95.780,13.888,host,host,96,110,15
8,7,110.444,6.699,Gouveia_Melo,Independent,111,118,8
9,,,3.61,(silence),,119,120,2


## 6 — Frame slideshow: top 3 segments per speaker

For each speaker (both candidates + host), shows the **3 longest audio segments**.
Every visual frame within the segment is shown — not just single-face ones.
Use the Play button or slider to scrub through and see what the camera was capturing.

In [12]:
import ipywidgets as widgets
from IPython.display import display as ipy_display, Audio as IPyAudio
import threading, base64 as _b64, wave as _wave_lib
import time as _time
from io import BytesIO as _BytesIO
from PIL import ImageDraw as _Draw
import scipy.io.wavfile as _wavfile

FRAMES_DIR = '../Frames'
AUDIO_DIR  = '../Audio'

def resolve_frame_path(pkl_path, frame_number):
    debate_folder = os.path.basename(os.path.dirname(pkl_path))
    if frame_number < 1000:
        return os.path.join(FRAMES_DIR, debate_folder, f'frame_{frame_number:03d}.jpg')
    return os.path.join(FRAMES_DIR, debate_folder, f'frame_{frame_number}.jpg')

# Load WAV once
_wav_path = os.path.join(AUDIO_DIR, f'{DEBATE}.wav')
if os.path.exists(_wav_path):
    _sr, _full_audio = _wavfile.read(_wav_path)
    if _full_audio.ndim == 2:
        _full_audio = _full_audio[:, 0]
    _full_audio = _full_audio.astype(np.float32) / 32768.0
    print(f'WAV loaded: sr={_sr} Hz, duration={len(_full_audio)//_sr}s')
else:
    _sr = None; _full_audio = None
    print(f'[!] WAV not found: {_wav_path}')

_resample = getattr(Image, 'Resampling', Image).LANCZOS

def _make_audio_html(chunk, sr):
    pcm = (chunk * 32767).astype(np.int16)
    buf = _BytesIO()
    with _wave_lib.open(buf, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())
    b64 = _b64.b64encode(buf.getvalue()).decode()
    return widgets.HTML(
        f'<audio controls style="width:900px" '
        f'src="data:audio/wav;base64,{b64}"></audio>'
    )

def _frame_bytes(frame_num, vis_row, target_w=900):
    img_path = resolve_frame_path(vis_row['Frame'], frame_num)
    try:
        img = Image.open(img_path).convert('RGB')
    except (FileNotFoundError, OSError):
        img = Image.new('RGB', (target_w, 506), (30, 30, 30))
    orig_w, orig_h = img.size
    target_h = int(orig_h * target_w / orig_w)
    sx, sy = target_w / orig_w, target_h / orig_h
    img  = img.resize((target_w, target_h), _resample)
    draw = _Draw.Draw(img)
    for fer in vis_row['Fer']:
        if not isinstance(fer, dict):
            continue
        x1, y1, x2, y2 = fer['bbox']
        draw.rectangle([int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy)],
                       outline='lime', width=3)
        draw.text((int(x1*sx)+2, max(int(y1*sy)-18, 0)),
                  fer.get('top_emotion', ''), fill='lime')
    buf = _BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

def _make_block(seg, all_frames, rank, speaker, frame_lookup):
    vrows = {f: frame_lookup.loc[f] for f in all_frames}

    header = widgets.HTML(
        f'<b>{speaker}</b> &nbsp;|&nbsp; {seg["party"]} &nbsp;|&nbsp; '
        f'Rank #{rank} by duration &nbsp;|&nbsp; '
        f't={seg["time stamp"]:.1f}s &nbsp; dur={seg["duration"]:.1f}s &nbsp;|&nbsp; '
        f'{len(all_frames)} frames'
    )
    img_w    = widgets.Image(format='jpeg', layout=widgets.Layout(width='900px'))
    slider   = widgets.IntSlider(value=0, min=0, max=len(all_frames)-1,
                                  description='Frame:', continuous_update=True,
                                  layout=widgets.Layout(width='500px'))
    play_btn = widgets.ToggleButton(value=False, description='▶ Play',
                                    button_style='success',
                                    layout=widgets.Layout(width='110px'))

    audio_w = None
    if _full_audio is not None:
        t0 = float(seg['time stamp'])
        s0  = int(t0 * _sr)
        s1  = int((t0 + float(seg['duration'])) * _sr)
        audio_w = _make_audio_html(_full_audio[s0:s1], _sr)

    _state = {'playing': False}

    def _show(idx):
        if 0 <= idx < len(all_frames):
            f = all_frames[idx]
            img_w.value = _frame_bytes(f, vrows[f])

    slider.observe(lambda c: _show(c['new']), names='value')

    def make_play_cb():
        def _cb(change):
            if change['new']:
                _state['playing'] = True
                play_btn.description = '⏸ Pause'
                start = slider.value
                def _run():
                    for i in range(start, len(all_frames)):
                        if not _state['playing']:
                            break
                        slider.value = i
                        _time.sleep(1.0)
                    if _state['playing']:
                        _state['playing'] = False
                        play_btn.value       = False
                        play_btn.description = '▶ Play'
                threading.Thread(target=_run, daemon=True).start()
            else:
                _state['playing']    = False
                play_btn.description = '▶ Play'
        return _cb

    play_btn.observe(make_play_cb(), names='value')
    _show(0)

    children = [header, img_w, widgets.HBox([play_btn, slider])]
    if audio_w is not None:
        children.append(audio_w)
    children.append(widgets.HTML("<hr style='margin:12px 0'>"))
    return widgets.VBox(children)

if not os.path.isdir(FRAMES_DIR):
    print(f"[!] '{FRAMES_DIR}/' not found.")
else:
    frame_lookup = data_visual.set_index('frame_number')
    cA, cB = extract_candidates(DEBATE)
    for speaker in [cA, cB, 'host']:
        top3 = debate_audio[debate_audio['speaker'] == speaker].nlargest(3, 'duration')
        for rank, (_, seg) in enumerate(top3.iterrows(), start=1):
            frames = sorted([f for f in seg['frames'] if f in frame_lookup.index])
            if frames:
                ipy_display(_make_block(seg, frames, rank, speaker, frame_lookup))


WAV loaded: sr=44100 Hz, duration=2048s


## 7 — Visual Candidate Classification

We bootstrap face identity from the audio labels already available, in three steps.

**Step 1 — Single-face frames as ground truth**  
When only 1 face is detected *and* the audio labels that moment as "Candidate X speaking", the camera is almost certainly on Candidate X. We collect those frames' facial landmarks as free labeled examples — no manual annotation needed. Even if ~15 % of single-face frames happen to show the *listener* instead, those outliers get diluted when we average across the full set.

**Step 2 — Build a face fingerprint per candidate**  
Each detected face carries 106 facial landmarks (eyes, nose, mouth, jawline, etc.). We normalize them by bounding-box size so the resulting vector captures face *shape* (identity) rather than screen position or scale. Averaging all collected vectors per candidate yields a stable **reference fingerprint**.

**Step 3 — Classify every face by nearest-neighbour matching**  
For any frame with 2 detected faces we compute the cosine distance from each face's landmark vector to both candidates' reference fingerprints and assign each face to the closer match. Single-face frames that fall outside any audio segment are also compared to the references instead of being left unlabelled.

In [22]:
from sklearn.metrics.pairwise import cosine_distances
from scipy.optimize import linear_sum_assignment
from collections import Counter
from PIL import ImageFont

# ── Edit first names here ─────────────────────────────────────────────────────
FIRST_NAMES = {
    cA: 'Catarina',
    cB: 'Henrique',
    'host': 'Host',
    'unknown': '?',
}

FACE_COLORS = {
    cA: '#FF4444',
    cB: '#4488FF',
    'host': '#FFDD00',
    'unknown': '#AAAAAA',
}

try:
    _label_font = ImageFont.truetype("arial.ttf", 20)
except (IOError, OSError):
    _label_font = ImageFont.load_default()

# ── Normalize 106 landmarks by bounding-box size ──────────────────────────────
def normalize_landmarks(face_dict):
    lm  = np.array(face_dict['landmarks'], dtype=float).reshape(106, 2)
    box = face_dict['bbox']
    w, h = box[2] - box[0], box[3] - box[1]
    if w <= 0 or h <= 0:
        return None
    lm -= np.mean(lm, axis=0)
    lm[:, 0] /= w
    lm[:, 1] /= h
    return lm.flatten()

# ── Step 1: collect landmarks from single-face segments ───────────────────────
print("Step 1 — collecting single-face reference frames...")
ref_pool = {cA: [], cB: [], 'host': []}

for _, seg in debate_audio[debate_audio['speaker'].isin([cA, cB, 'host'])].iterrows():
    for f in seg['frames']:
        if f not in frame_lookup.index:
            continue
        fer = frame_lookup.loc[f, 'Fer']
        if not isinstance(fer, list) or len(fer) != 1:
            continue
        face = fer[0]
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is not None:
            ref_pool[seg['speaker']].append(lm)

for person, vecs in ref_pool.items():
    print(f"  {FIRST_NAMES.get(person, person):12s}: {len(vecs)} reference frames")

# ── Step 2: mean fingerprint per person ──────────────────────────────────────
print("\nStep 2 — building face fingerprints...")
ref = {p: np.mean(np.stack(v), axis=0) for p, v in ref_pool.items() if v}

_persons_all = [c for c in [cA, cB, 'host'] if c in ref]

# ── Optimal one-to-one assignment using the Hungarian algorithm ───────────────
def assign_faces(fer, persons):
    """Build a cost matrix and find the best unique face→person assignment."""
    n_f, n_p = len(fer), len(persons)
    cost = np.ones((n_f, n_p))            # default = max cost
    for i, face in enumerate(fer):
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is None:
            continue
        for j, person in enumerate(persons):
            if person in ref:
                cost[i, j] = cosine_distances(
                    lm.reshape(1, -1), ref[person].reshape(1, -1)
                )[0, 0]
    row_ind, col_ind = linear_sum_assignment(cost)
    labels = ['unknown'] * n_f
    for r, c in zip(row_ind, col_ind):
        labels[r] = persons[c]
    return labels

# ── Step 3: classify every frame ──────────────────────────────────────────────
print("\nStep 3 — classifying faces in all frames...")
frame_visual_labels = {}

for _, row in data_visual.iterrows():
    f   = row['frame_number']
    fer = row['Fer']
    if not isinstance(fer, list) or len(fer) == 0:
        frame_visual_labels[f] = []
        continue

    seg_match = debate_audio[
        (debate_audio['frame_start'] <= f) &
        (debate_audio['frame_end']   >= f) &
        (debate_audio['speaker'].isin([cA, cB, 'host']))
    ]
    audio_spk = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else None

    if len(fer) == 1:
        # single face: trust audio label directly; use classifier only when silent
        if audio_spk:
            frame_visual_labels[f] = [audio_spk]
        else:
            frame_visual_labels[f] = assign_faces(fer, _persons_all)
    elif len(fer) == 2:
        # two faces on screen → always the two candidates
        frame_visual_labels[f] = assign_faces(fer, [cA, cB])
    else:
        # three or more faces → all three people
        frame_visual_labels[f] = assign_faces(fer, _persons_all)

print("Done.\n")
print("Visual label distribution (face detections across all frames):")
all_labels = [l for labels in frame_visual_labels.values() for l in labels]
for label, count in Counter(all_labels).most_common():
    print(f"  {FIRST_NAMES.get(label, label):12s}: {count:4d} face detections")

Step 1 — collecting single-face reference frames...
  Catarina    : 404 reference frames
  Henrique    : 350 reference frames
  Host        : 111 reference frames

Step 2 — building face fingerprints...

Step 3 — classifying faces in all frames...
Done.

Visual label distribution (face detections across all frames):
  Henrique    : 1513 face detections
  Catarina    : 1479 face detections
  Host        :  270 face detections
  ?           :    1 face detections


In [23]:
## 10 — Option 2: SVM classifier on landmark features

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler as _SS2
from scipy.optimize import linear_sum_assignment as _lsa2
from collections import Counter as _C2

# ── Build training set from ref_pool (computed in cell 7) ─────────────────────
_X2, _y2 = [], []
for person, vecs in ref_pool.items():
    for v in vecs:
        _X2.append(v)
        _y2.append(person)

_X2 = np.stack(_X2)
_y2 = np.array(_y2)

print("Training set:")
for person in [cA, cB, 'host']:
    n = (_y2 == person).sum()
    print(f"  {FIRST_NAMES.get(person, person):12s}: {n} samples")

# ── Scale + train SVM (probability=True enables cost-matrix assignment) ────────
_scaler_svm = _SS2()
_X2_scaled  = _scaler_svm.fit_transform(_X2)

_clf_svm = SVC(kernel='rbf', C=10, gamma='scale',
               class_weight='balanced', probability=True)
_clf_svm.fit(_X2_scaled, _y2)
print(f"\nSVM trained — {_clf_svm.support_vectors_.shape[0]} support vectors")
print(f"Classes order: {list(_clf_svm.classes_)}")

# ── Optimal assignment using SVM probabilities as costs ───────────────────────
def assign_faces_svm(fer, persons):
    """Cost = 1 − P(person|face); Hungarian algorithm gives unique assignment."""
    n_f, n_p = len(fer), len(persons)
    cost = np.ones((n_f, n_p))
    classes = list(_clf_svm.classes_)
    for i, face in enumerate(fer):
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is None:
            continue
        probs = _clf_svm.predict_proba(_scaler_svm.transform(lm.reshape(1, -1)))[0]
        for j, person in enumerate(persons):
            if person in classes:
                cost[i, j] = 1.0 - probs[classes.index(person)]
    row_ind, col_ind = _lsa2(cost)
    labels = ['unknown'] * n_f
    for r, c in zip(row_ind, col_ind):
        labels[r] = persons[c]
    return labels

# ── Classify every frame ──────────────────────────────────────────────────────
frame_visual_labels_svm = {}

for _, row in data_visual.iterrows():
    f   = row['frame_number']
    fer = row['Fer']
    if not isinstance(fer, list) or len(fer) == 0:
        frame_visual_labels_svm[f] = []
        continue

    seg_match = debate_audio[
        (debate_audio['frame_start'] <= f) &
        (debate_audio['frame_end']   >= f) &
        (debate_audio['speaker'].isin([cA, cB, 'host']))
    ]
    audio_spk = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else None

    if len(fer) == 1:
        if audio_spk:
            frame_visual_labels_svm[f] = [audio_spk]
        else:
            frame_visual_labels_svm[f] = assign_faces_svm(fer, _persons_all)
    elif len(fer) == 2:
        frame_visual_labels_svm[f] = assign_faces_svm(fer, [cA, cB])
    else:
        frame_visual_labels_svm[f] = assign_faces_svm(fer, _persons_all)

print("\nOption 2 — SVM label distribution:")
_all_svm = [l for labels in frame_visual_labels_svm.values() for l in labels]
for label, count in _C2(_all_svm).most_common():
    print(f"  {FIRST_NAMES.get(label, label):12s}: {count:4d} face detections")

Training set:
  Catarina    : 404 samples
  Henrique    : 350 samples
  Host        : 111 samples

SVM trained — 295 support vectors
Classes order: [np.str_('Gouveia_Melo'), np.str_('Martins'), np.str_('host')]

Option 2 — SVM label distribution:
  Henrique    : 1515 face detections
  Catarina    : 1472 face detections
  Host        :  275 face detections
  ?           :    1 face detections


In [24]:
## 11 — Option 3: K-means clustering on ALL face detections

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler as _SS3
from scipy.optimize import linear_sum_assignment as _lsa3
from collections import Counter as _C3

# ── Extract landmarks from ALL face detections across all frames ──────────────
print("Extracting landmarks from all frames...")
_all_lm_vecs = []
_all_lm_meta = []   # (frame_number, face_index, audio_speaker, n_faces)

for _, row in data_visual.iterrows():
    f   = row['frame_number']
    fer = row['Fer']
    if not isinstance(fer, list) or len(fer) == 0:
        continue

    seg_match = debate_audio[
        (debate_audio['frame_start'] <= f) &
        (debate_audio['frame_end']   >= f) &
        (debate_audio['speaker'].isin([cA, cB, 'host']))
    ]
    audio_spk = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else None

    for i, face in enumerate(fer):
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is None:
            continue
        _all_lm_vecs.append(lm)
        _all_lm_meta.append((f, i, audio_spk, len(fer)))

_X_all   = np.stack(_all_lm_vecs)
_meta_df = pd.DataFrame(_all_lm_meta, columns=['frame', 'face_idx', 'audio_speaker', 'n_faces'])
print(f"Total face detections with valid landmarks: {len(_X_all)}")

# ── K-means k=3 ───────────────────────────────────────────────────────────────
print("Running K-means (k=3)...")
_scaler_km = _SS3()
_X_km      = _scaler_km.fit_transform(_X_all)
_km        = KMeans(n_clusters=3, random_state=42, n_init=10)
_meta_df['cluster'] = _km.fit_predict(_X_km)
print("Done.\n")

# ── Name clusters via majority vote (single-face + audio frames only) ─────────
_single = _meta_df[(_meta_df['n_faces'] == 1) & _meta_df['audio_speaker'].notna()]
_votes  = _single.groupby(['cluster', 'audio_speaker']).size().unstack(fill_value=0)
print("Cluster vote matrix:")
print(_votes.to_string())

_persons_left  = [cA, cB, 'host']
_clusters_left = list(range(3))
_cluster_map   = {}

for _ in range(3):
    best = (-1, None, None)
    for c in _clusters_left:
        for p in _persons_left:
            cnt = int(_votes.loc[c, p]) if (c in _votes.index and p in _votes.columns) else 0
            if cnt > best[0]:
                best = (cnt, c, p)
    _, bc, bp = best
    _cluster_map[bc] = bp
    _clusters_left.remove(bc)
    _persons_left.remove(bp)

print("\nCluster assignments:")
for c, p in _cluster_map.items():
    print(f"  Cluster {c} → {FIRST_NAMES.get(p, p)}")

# ── Optimal assignment per frame using distance to cluster centroids ───────────
# Map person → cluster index → centroid in scaled space
_person_to_centroid = {
    p: _km.cluster_centers_[c] for c, p in _cluster_map.items()
}

def assign_faces_km(fer, persons):
    """Cost = Euclidean distance to cluster centroid; Hungarian gives unique assignment."""
    n_f, n_p = len(fer), len(persons)
    cost = np.ones((n_f, n_p)) * 1e6
    for i, face in enumerate(fer):
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is None:
            continue
        lm_scaled = _scaler_km.transform(lm.reshape(1, -1))[0]
        for j, person in enumerate(persons):
            if person in _person_to_centroid:
                cost[i, j] = np.linalg.norm(lm_scaled - _person_to_centroid[person])
    row_ind, col_ind = _lsa3(cost)
    labels = ['unknown'] * n_f
    for r, c in zip(row_ind, col_ind):
        labels[r] = persons[c]
    return labels

# ── Assign labels to every frame ──────────────────────────────────────────────
frame_visual_labels_kmeans = {}

for _, row in data_visual.iterrows():
    f   = row['frame_number']
    fer = row['Fer']
    if not isinstance(fer, list) or len(fer) == 0:
        frame_visual_labels_kmeans[f] = []
        continue

    seg_match = debate_audio[
        (debate_audio['frame_start'] <= f) &
        (debate_audio['frame_end']   >= f) &
        (debate_audio['speaker'].isin([cA, cB, 'host']))
    ]
    audio_spk = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else None

    if len(fer) == 1 and audio_spk:
        frame_visual_labels_kmeans[f] = [audio_spk]
    elif len(fer) == 2:
        frame_visual_labels_kmeans[f] = assign_faces_km(fer, [cA, cB])
    else:
        frame_visual_labels_kmeans[f] = assign_faces_km(fer, _persons_all)

print("\nOption 3 — K-means label distribution:")
_all_km = [l for labels in frame_visual_labels_kmeans.values() for l in labels]
for label, count in _C3(_all_km).most_common():
    print(f"  {FIRST_NAMES.get(label, label):12s}: {count:4d} face detections")

Extracting landmarks from all frames...
Total face detections with valid landmarks: 3263
Running K-means (k=3)...
Done.

Cluster vote matrix:
audio_speaker  Gouveia_Melo  Martins  host
cluster                                   
0                       314       22    43
1                        19      289    58
2                        10       87     6

Cluster assignments:
  Cluster 0 → Henrique
  Cluster 1 → Catarina
  Cluster 2 → Host

Option 3 — K-means label distribution:
  Henrique    : 1515 face detections
  Catarina    : 1475 face detections
  Host        :  272 face detections
  ?           :    1 face detections


In [26]:
## 9 — Option 1b: Original + spatial heuristics
#
# Same cosine-distance fingerprints as cell 7, but with two spatial rules:
#   1. Two faces on screen → host is allowed as one of them.
#      A 2×3 cost matrix is built (2 faces × [cA, cB, host]); the Hungarian
#      algorithm picks whichever two-person pairing minimises total distance.
#   2. Three faces on screen → the middle face by x-position is the host.
#      The remaining two faces are then assigned to [cA, cB] optimally.

from scipy.optimize import linear_sum_assignment as _lsa_h
from collections import Counter as _Ch

def _assign_h(fer, persons):
    """Hungarian assignment of len(fer) faces to persons (rectangular cost matrix ok)."""
    n_f, n_p = len(fer), len(persons)
    cost = np.ones((n_f, n_p))
    for i, face in enumerate(fer):
        if not isinstance(face, dict) or len(face.get('landmarks', [])) != 106:
            continue
        lm = normalize_landmarks(face)
        if lm is None:
            continue
        for j, person in enumerate(persons):
            if person in ref:
                cost[i, j] = cosine_distances(
                    lm.reshape(1, -1), ref[person].reshape(1, -1)
                )[0, 0]
    row_ind, col_ind = _lsa_h(cost)
    labels = ['unknown'] * n_f
    for r, c in zip(row_ind, col_ind):
        labels[r] = persons[c]
    return labels

frame_visual_labels_heuristic = {}

for _, row in data_visual.iterrows():
    f   = row['frame_number']
    fer = row['Fer']
    if not isinstance(fer, list) or len(fer) == 0:
        frame_visual_labels_heuristic[f] = []
        continue

    seg_match = debate_audio[
        (debate_audio['frame_start'] <= f) &
        (debate_audio['frame_end']   >= f) &
        (debate_audio['speaker'].isin([cA, cB, 'host']))
    ]
    audio_spk = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else None

    if len(fer) == 1:
        if audio_spk:
            frame_visual_labels_heuristic[f] = [audio_spk]
        else:
            frame_visual_labels_heuristic[f] = _assign_h(fer, _persons_all)

    elif len(fer) == 2:
        # heuristic 1: build 2×3 cost matrix — host can win one of the two spots
        frame_visual_labels_heuristic[f] = _assign_h(fer, _persons_all)

    else:
        # heuristic 2: middle face by x-center = host; assign the rest to candidates
        x_centers  = [
            (face['bbox'][0] + face['bbox'][2]) / 2
            if isinstance(face, dict) and 'bbox' in face else 0
            for face in fer
        ]
        sorted_i   = sorted(range(len(fer)), key=lambda i: x_centers[i])
        middle_idx = sorted_i[len(sorted_i) // 2]

        labels = ['unknown'] * len(fer)
        labels[middle_idx] = 'host'

        other_idx    = [i for i in range(len(fer)) if i != middle_idx]
        other_labels = _assign_h([fer[i] for i in other_idx], [cA, cB])
        for i, lbl in zip(other_idx, other_labels):
            labels[i] = lbl

        frame_visual_labels_heuristic[f] = labels

print("Option 1b — heuristic label distribution:")
_all_h = [l for labels in frame_visual_labels_heuristic.values() for l in labels]
for label, count in _Ch(_all_h).most_common():
    print(f"  {FIRST_NAMES.get(label, label):12s}: {count:4d} face detections")

Option 1b — heuristic label distribution:
  Henrique    : 1509 face detections
  Catarina    : 1318 face detections
  Host        :  435 face detections
  ?           :    1 face detections


In [27]:
## 8 — Labeled frame browser (all options)

import threading as _thr
import time as _tm

def _get_label_set(option):
    try:
        return {'original':  frame_visual_labels,
                'heuristic': frame_visual_labels_heuristic,
                'svm':       frame_visual_labels_svm,
                'kmeans':    frame_visual_labels_kmeans}[option]
    except NameError:
        return {}

def _frame_bytes_labeled(frame_num, vis_row, option='original', target_w=900):
    selected = _get_label_set(option)

    img_path = resolve_frame_path(vis_row['Frame'], frame_num)
    try:
        img = Image.open(img_path).convert('RGB')
    except (FileNotFoundError, OSError):
        img = Image.new('RGB', (target_w, 506), (30, 30, 30))

    orig_w, orig_h = img.size
    target_h = int(orig_h * target_w / orig_w)
    sx, sy   = target_w / orig_w, target_h / orig_h
    img  = img.resize((target_w, target_h), _resample)
    draw = _Draw.Draw(img)

    fer         = vis_row['Fer'] if isinstance(vis_row['Fer'], list) else []
    face_labels = selected.get(frame_num, [])

    for i, face in enumerate(fer):
        if not isinstance(face, dict):
            continue
        x1, y1, x2, y2 = face['bbox']
        label      = face_labels[i] if i < len(face_labels) else 'unknown'
        first_name = FIRST_NAMES.get(label, label)
        color      = FACE_COLORS.get(label, '#AAAAAA')
        bx1, by1   = int(x1*sx), int(y1*sy)
        bx2, by2   = int(x2*sx), int(y2*sy)
        draw.rectangle([bx1, by1, bx2, by2], outline=color, width=4)
        try:
            tb = _label_font.getbbox(first_name)
            tw, th = tb[2]-tb[0], tb[3]-tb[1]
        except AttributeError:
            tw, th = len(first_name)*9, 16
        tx, ty = bx1, max(by1-th-8, 0)
        draw.rectangle([tx, ty, tx+tw+10, ty+th+6], fill=color)
        draw.text((tx+5, ty+3), first_name, fill='black', font=_label_font)

    seg_match   = debate_audio[
        (debate_audio['frame_start'] <= frame_num) &
        (debate_audio['frame_end']   >= frame_num)
    ]
    audio_label = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else 'silence'
    info = (f"Frame {frame_num}  |  [{option}]  |  "
            f"Audio: {FIRST_NAMES.get(audio_label, audio_label)} speaking")
    try:
        ib = _label_font.getbbox(info)
        iw, ih = ib[2]-ib[0], ib[3]-ib[1]
    except AttributeError:
        iw, ih = len(info)*9, 16
    draw.rectangle([0, 0, iw+16, ih+10], fill=(0, 0, 0))
    draw.text((8, 5), info, fill='white', font=_label_font)

    buf = _BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

# ── Widgets ───────────────────────────────────────────────────────────────────
_all_frames = sorted(data_visual['frame_number'].tolist())
_idx_of     = {f: i for i, f in enumerate(_all_frames)}

_img_w   = widgets.Image(format='jpeg', layout=widgets.Layout(width='900px'))
_slider  = widgets.IntSlider(
    value=_all_frames[0], min=_all_frames[0], max=_all_frames[-1],
    description='Frame:', continuous_update=True,
    layout=widgets.Layout(width='560px'),
)
_playbtn = widgets.ToggleButton(
    value=False, description='▶ Play',
    button_style='success', layout=widgets.Layout(width='110px'),
)
_option_dd = widgets.Dropdown(
    options=[('Original (centroid)',       'original'),
             ('Option 1b — + heuristics',  'heuristic'),
             ('Option 2 — SVM',            'svm'),
             ('Option 3 — K-means',        'kmeans')],
    value='original',
    description='Method:',
    layout=widgets.Layout(width='290px'),
)
_pstate = {'on': False}

def _show(f):
    if f in frame_lookup.index:
        _img_w.value = _frame_bytes_labeled(f, frame_lookup.loc[f],
                                            option=_option_dd.value)

_option_dd.observe(lambda c: _show(_slider.value), names='value')
_slider.observe(lambda c: _show(c['new']), names='value')

def _play_cb(change):
    if change['new']:
        _pstate['on'] = True
        _playbtn.description = '⏸ Pause'
        start_i = _idx_of.get(_slider.value, 0)
        def _run():
            for f in _all_frames[start_i:]:
                if not _pstate['on']:
                    break
                _slider.value = f
                _tm.sleep(1.0)
            _pstate['on'] = False
            _playbtn.value = False
            _playbtn.description = '▶ Play'
        _thr.Thread(target=_run, daemon=True).start()
    else:
        _pstate['on'] = False
        _playbtn.description = '▶ Play'

_playbtn.observe(_play_cb, names='value')
_show(_all_frames[0])

legend = (
    f'<b>Labeled frame browser — {DEBATE}</b> &nbsp;&nbsp;'
    f'<span style="color:{FACE_COLORS[cA]}">■ {FIRST_NAMES[cA]}</span> &nbsp;'
    f'<span style="color:{FACE_COLORS[cB]}">■ {FIRST_NAMES[cB]}</span> &nbsp;'
    f'<span style="color:{FACE_COLORS["host"]}">■ Host</span>'
)
ipy_display(widgets.VBox([
    widgets.HTML(legend),
    _img_w,
    widgets.HBox([_playbtn, _slider, _option_dd]),
]))

In [ ]:
## 12 — Export labeled video

import cv2

# ── Choose which classification to export ─────────────────────────────────────
#   'original'   → centroid cosine distance        (cell 7)
#   'heuristic'  → original + spatial heuristics   (cell 9)
#   'svm'        → SVM classifier                  (cell 10)
#   'kmeans'     → K-means clustering              (cell 11)
OPTION = 'original'

_label_options = {
    'original':  frame_visual_labels,
    'heuristic': frame_visual_labels_heuristic,
    'svm':       frame_visual_labels_svm,
    'kmeans':    frame_visual_labels_kmeans,
}
_selected_labels = _label_options[OPTION]
OUTPUT_PATH      = f'../labeled_{DEBATE}_{OPTION}.mp4'
FPS              = 1      # raise to e.g. 5 for a faster time-lapse
TARGET_W         = 900

# ── Render one frame with the selected labels ─────────────────────────────────
def _render_frame(frame_num, vis_row):
    img_path = resolve_frame_path(vis_row['Frame'], frame_num)
    try:
        img = Image.open(img_path).convert('RGB')
    except (FileNotFoundError, OSError):
        img = Image.new('RGB', (TARGET_W, 506), (30, 30, 30))
    orig_w, orig_h = img.size
    target_h = int(orig_h * TARGET_W / orig_w)
    sx, sy   = TARGET_W / orig_w, target_h / orig_h
    img  = img.resize((TARGET_W, target_h), _resample)
    draw = _Draw.Draw(img)

    fer         = vis_row['Fer'] if isinstance(vis_row['Fer'], list) else []
    face_labels = _selected_labels.get(frame_num, [])

    for i, face in enumerate(fer):
        if not isinstance(face, dict):
            continue
        x1, y1, x2, y2 = face['bbox']
        label      = face_labels[i] if i < len(face_labels) else 'unknown'
        first_name = FIRST_NAMES.get(label, label)
        color      = FACE_COLORS.get(label, '#AAAAAA')
        bx1, by1   = int(x1*sx), int(y1*sy)
        bx2, by2   = int(x2*sx), int(y2*sy)
        draw.rectangle([bx1, by1, bx2, by2], outline=color, width=4)
        try:
            tb = _label_font.getbbox(first_name)
            tw, th = tb[2]-tb[0], tb[3]-tb[1]
        except AttributeError:
            tw, th = len(first_name)*9, 16
        tx, ty = bx1, max(by1-th-8, 0)
        draw.rectangle([tx, ty, tx+tw+10, ty+th+6], fill=color)
        draw.text((tx+5, ty+3), first_name, fill='black', font=_label_font)

    seg_match   = debate_audio[
        (debate_audio['frame_start'] <= frame_num) &
        (debate_audio['frame_end']   >= frame_num)
    ]
    audio_label = seg_match.iloc[0]['speaker'] if len(seg_match) > 0 else 'silence'
    info = (f"Frame {frame_num}  |  [{OPTION}]  |  "
            f"Audio: {FIRST_NAMES.get(audio_label, audio_label)} speaking")
    try:
        ib = _label_font.getbbox(info)
        iw, ih = ib[2]-ib[0], ib[3]-ib[1]
    except AttributeError:
        iw, ih = len(info)*9, 16
    draw.rectangle([0, 0, iw+16, ih+10], fill=(0, 0, 0))
    draw.text((8, 5), info, fill='white', font=_label_font)

    buf = _BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

# ── Detect output frame height from the first frame ───────────────────────────
_test    = Image.open(_BytesIO(_render_frame(_all_frames[0], frame_lookup.loc[_all_frames[0]])))
TARGET_H = _test.height
print(f"Option : {OPTION}")
print(f"Size   : {TARGET_W}x{TARGET_H}  |  {len(_all_frames)} frames  |  {FPS} fps")
print(f"Output : {OUTPUT_PATH}\n")

# ── Write video ───────────────────────────────────────────────────────────────
_fourcc = cv2.VideoWriter_fourcc(*'mp4v')
_writer = cv2.VideoWriter(OUTPUT_PATH, _fourcc, FPS, (TARGET_W, TARGET_H))

for i, f in enumerate(_all_frames):
    if i % 200 == 0:
        print(f"  {i}/{len(_all_frames)} frames written...")
    if f not in frame_lookup.index:
        continue
    _raw = _render_frame(f, frame_lookup.loc[f])
    _arr = np.array(Image.open(_BytesIO(_raw)).convert('RGB'))
    _bgr = cv2.cvtColor(_arr, cv2.COLOR_RGB2BGR)
    _writer.write(_bgr)

_writer.release()
print(f"\nDone — {OUTPUT_PATH}")